# Autoresearch

An agent edits one file, measures, keeps or discards — and repeats without you.

*You are the slow part of research, not the thinking.*

> Faithful to [karpathy/autoresearch](https://github.com/karpathy/autoresearch): one loop.

`solve(n)` counts the primes below `n`, starting from the sieve everyone writes first. There is
no closed form to jump to, so the loop has to climb — §5 measures how far it could have gone.

## 0. Setup

Reading and writing are separate permissions — this split *is* the design:

| path | reads | writes |
|---|---|---|
| `program.md` | **yes** — every prompt | no |
| `src/` | **yes** | **yes** — on a win |
| `harness/` | no | no |

In [1]:
import os
import re
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

from harness.measure import measure   # protected/ — never enters a prompt

load_dotenv(find_dotenv(usecwd=True))
client = OpenAI(
    api_key=os.environ["DEEPINFRA_API_KEY"],
    base_url="https://api.deepinfra.com/v1/openai",
)
MODEL = "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"

SOLVE = Path("src/solve.py")   # the only file the agent may rewrite
PROGRAM = Path("program.md").read_text()
MAX_EXPERIMENTS, PATIENCE = 15, 8
MIN_GAIN = 0.95   # a win must beat best by 5% — under that, the timer can't tell

## 1. The boundary — load both and look

The agent reads `program.md` and `src/`. It never sees `harness/`, so the verifier can't be
argued with.

In [2]:
EDITABLE  = sorted(f for f in Path("src").iterdir() if f.is_file())
PROTECTED = sorted(f for f in Path("harness").iterdir() if f.is_file())

for label, paths, seen in [("EDITABLE  src/", EDITABLE, "the agent rewrites these"),
                           ("PROTECTED harness/", PROTECTED, "never enters a prompt")]:
    print(f"{'='*70}\n{label}  — {seen}\n{'='*70}")
    for f in paths:
        print(f"--- {f} ---")
        print(f.read_text().rstrip(), "\n")

EDITABLE  src/  — the agent rewrites these
--- src/solve.py ---
# EDITABLE — autoresearch rewrites this in place. Karpathy's train.py.


def solve(n):
    sieve = [True] * n
    sieve[0] = sieve[1] = False
    i = 2
    while i * i < n:
        if sieve[i]:
            for j in range(i * i, n, i):
                sieve[j] = False
        i += 1
    return sum(sieve) 

PROTECTED harness/  — never enters a prompt
--- harness/cases.json ---
{
  "_note": "count of primes below n — n is exclusive. pi(7)=3 counts 2,3,5 and NOT 7. This is the spec the model keeps getting wrong, and only a prime n catches it.",
  "tests": [[7, 3], [10, 4], [100, 25], [1000, 168], [10000, 1229]],
  "workload": 2000000,
  "expected": 148933
} 

--- harness/measure.py ---
"""PROTECTED — never enters a prompt. Karpathy's `prepare.py`.

An agent that can edit the verifier optimises the verifier: it deletes the failing
case instead of passing it.
"""

import json
import time
from pathlib import Path

_CASES = json.l

## 2. The prompt

Everything the agent gets: `program.md`, the editable file, the last measurement. Nothing else.

In [3]:
def propose(src, seconds, note):
    """The only thing the agent ever sees: program.md + solve.py + the last measurement."""
    msg = (f"{PROGRAM}\n\n"
           f"src/solve.py:\n```python\n{src}\n```\n"
           f"Current: {seconds*1000:.3f} ms. Last result: {note}\n"
           f"Propose ONE change. Reply with only the new `def solve(n):` in a ```python block.")
    out = client.chat.completions.create(
        model=MODEL, max_tokens=700, messages=[{"role": "user", "content": msg}],
    ).choices[0].message.content
    m = re.search(r"```python\n(.*?)```", out, re.S)
    return m.group(1).strip() if m else None

## 3. The loop

Measure, then write only on a win — so `src/solve.py` only ever holds the best. A win must
clear 5%: `measure` already takes the min of 3 runs, and anything smaller is still the timer.

In [4]:
def autoresearch():
    best = SOLVE.read_text()
    best_s, note = measure(best)
    stale = 0

    for i in range(1, MAX_EXPERIMENTS + 1):
        cand = propose(best, best_s, note)
        secs, note = measure(cand) if cand else (None, "no code block")
        kept = secs is not None and secs < best_s * MIN_GAIN   # 5% — not noise

        if kept:
            SOLVE.write_text(cand)          # commit: the file only ever holds the best
            best, best_s, stale = cand, secs, 0
        else:
            stale += 1                      # discard: nothing was written, nothing to undo

        print(f"  {i:>2} {('%.1f ms' % (secs*1000)) if secs else '—':>10}  "
              f"{'KEEP ' if kept else 'discard'}  best={best_s*1000:>7.1f} ms  {note[:46]}")

        if stale >= PATIENCE:
            print(f"  stop: no improvement in {PATIENCE} experiments")
            break
    return best, best_s

## 4. Run

`git diff src/` afterwards to see what it did.

In [5]:
base = SOLVE.read_text()
base_s, _ = measure(base)
print(f"baseline {base_s*1000:.1f} ms\n")

best, best_s = autoresearch()
print(f"\nbaseline {base_s*1000:.1f} ms -> best {best_s*1000:.1f} ms  ({base_s/best_s:.1f}x)")
print(best)

baseline 69.4 ms



   1    69.4 ms  discard  best=   67.6 ms  ok


   2    67.4 ms  discard  best=   67.6 ms  ok


   3    67.8 ms  discard  best=   67.6 ms  ok


   4    67.5 ms  discard  best=   67.6 ms  ok


...

  11    37.2 ms  discard  best=   26.3 ms  ok


  12    25.8 ms  discard  best=   26.3 ms  ok


  13    26.2 ms  discard  best=   26.3 ms  ok


  14    26.9 ms  discard  best=   26.3 ms  ok


  15    31.5 ms  discard  best=   26.3 ms  ok

baseline 69.4 ms -> best 26.3 ms  (2.6x)
def solve(n):
    if n < 3:
        return 0
    sieve = [True] * (n // 2)
    sieve[0] = False
    i = 1
    while i * (i + 1) * 2 < n:
        if sieve[i]:
            for j in range((i * (i + 1) * 2), n // 2, i * 2 + 1):
                sieve[j] = False
        i += 1
    return sum(sieve) + 1


## 5. The ceiling — what was on the table

The loop's speedup means nothing without one. Each rung is hand-written and scored by the same
`measure`, so the gate proves it correct before it is timed. Nothing here enters a prompt.

In [6]:
RUNGS = {
"1 + slice assignment": """
def solve(n):
    sieve = bytearray([1]) * n
    sieve[0] = sieve[1] = 0
    i = 2
    while i * i < n:
        if sieve[i]:
            sieve[i * i::i] = bytearray(len(range(i * i, n, i)))   # inner loop -> C
        i += 1
    return sum(sieve)
""",
"2 + odds only": """
def solve(n):
    sieve = bytearray([1]) * (n // 2)        # index i <-> the odd number 2i+1
    sieve[0] = 0
    i = 3
    while i * i < n:
        if sieve[i // 2]:
            start = i * i // 2
            sieve[start::i] = bytearray(len(range(start, n // 2, i)))
        i += 2
    return sum(sieve) + 1                    # +1 for 2, the only even prime
""",
}

print(f"{'0 the sieve in src/ — baseline':<32} {base_s*1000:>6.1f} ms\n")
for name, src in RUNGS.items():
    s, note = measure(src)                   # same gate: correct before it is timed
    print(f"{name:<32} {s*1000:>6.1f} ms  {base_s/s:>5.1f}x  {note}")
print(f"\nthe loop stopped at {base_s/best_s:.1f}x")

0 the sieve in src/ — baseline     69.4 ms

1 + slice assignment                6.8 ms   10.1x  ok
2 + odds only                       3.3 ms   20.9x  ok

the loop stopped at 2.6x


## Key findings

The run above, `MAX_EXPERIMENTS=15, PATIENCE=8`, Llama-4-Maverick:

```
baseline 69.4 ms -> best 26.3 ms   (2.6x)
  1-7           discard — nothing, seven times
   8   26.3 ms  KEEP     <- odds only, the one idea it had
  9-15          discard
```

**The win was experiment 8.** Seven proposals found nothing first — a human hand-running these
stops at three. Not getting bored is the whole argument for the loop.

**It never proposed slice assignment.** §5 measures `10.1x` for that one line and `20.9x` with
both rungs; the loop took `2.6x`, an eighth of what was there. Fifteen experiments were plenty —
the model, not the loop, was the ceiling.

The discards are the noise fix working. Experiment 12 measured `25.8 ms` against a `26.3 ms`
best and was thrown away: 1.9% is under the 5% gate, and min-of-3 still leaves a 1-3% spread.
Before the fix, three such candidates were committed as wins.

## Weaknesses

| Weakness | What happens | Fix |
|---|---|---|
| **The model is the ceiling** | Maverick found odds-only at experiment 8 (`2.6x`) and never proposed slice assignment — one line, `10.1x`, and §5 proves it passes the same gate. The other 14 experiments were variations | Use a model that has the idea |
| **No memory** | Each proposal sees only the best and the last note, so it re-proposes what it just tried: 1–7 all landed within 3% of the baseline, 9–15 within 3% of experiment 8 | Pass the log |
| **The gate is hand-tuned** | Min-of-3 still leaves a 1–3% spread, and `MIN_GAIN=0.95` sits just above it. Both were picked for this workload and this machine — on a faster box, or at a smaller `n`, a real win could fall under the gate | Re-time one candidate N times and set the gate above the measured spread |
| **The boundary is convention** | `exec` runs model code in-process with write access to `harness/` | Subprocess, read-only mount, timeout |
| **The metric is the objective** | It optimises `measure`'s return, not your intent. Correctness is pass/fail — no partial credit | Verify the verifier first. `n=7` is in `cases.json` because every other case is composite and cannot see a `<= n` off-by-one |
| **No rollback** | The first win overwrites the baseline in place | Commit before running — this ate the scaffold twice while the module was being built |
| **One loop, by design** | The strategy never changes; nothing notices that experiments 1–7 were going nowhere | none — a second loop is a different mechanism |